In [1]:
from Bio import SeqIO
from glob import glob
import re
import os
import pandas as pd

In [2]:
RXLR_pattern =  re.compile("[RQGH].LR")
hdRXLR_pattern = re.compile("[RKHGQ].{1,2}[LMYFIV][RNK]")
EER_pattern = re.compile("[ED][ED][KR]")

In [3]:
%cd ../analysis/1.RXLR_prediction/1-2.pattern_match/

/var2/Works/junhyeong/RXLR_landscape/analysis/1.RXLR_prediction/1-2.pattern_match


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
processed_entries = glob("../1-1.signalp/results.parse/*_signalp.fasta")

In [10]:
processed_entries

['../1-1.signalp/results.parse/Phycap_signalp.fasta',
 '../1-1.signalp/results.parse/Physoj_signalp.fasta',
 '../1-1.signalp/results.parse/Pytult_signalp.fasta',
 '../1-1.signalp/results.parse/Hyaara_signalp.fasta',
 '../1-1.signalp/results.parse/Pytaph_signalp.fasta',
 '../1-1.signalp/results.parse/Phyinf_signalp.fasta',
 '../1-1.signalp/results.parse/Phypal_signalp.fasta',
 '../1-1.signalp/results.parse/Phymeg_signalp.fasta',
 '../1-1.signalp/results.parse/Phycin_signalp.fasta',
 '../1-1.signalp/results.parse/Phycac_signalp.fasta',
 '../1-1.signalp/results.parse/Phyram_signalp.fasta',
 '../1-1.signalp/results.parse/Brelac_signalp.fasta',
 '../1-1.signalp/results.parse/Phylat_signalp.fasta',
 '../1-1.signalp/results.parse/Phypar_signalp.fasta',
 '../1-1.signalp/results.parse/Phyker_signalp.fasta']

In [21]:
df = pd.DataFrame()

In [22]:
for fasta in processed_entries:
    out = os.path.basename(fasta)
    spp = out.replace("_signalp.fasta", "")
    out = out.replace("_signalp", "_signalp_regex")

    out = os.path.join("results", out)
    
    if os.path.exists(out):
       continue 

    print(out)
    
    with open(os.path.join(out), "w") as f:
        for record in SeqIO.parse(fasta, "fasta"):
            entry = record.id
            seq = str(record.seq)

            df.loc[entry, "seq"] = seq
            df.loc[entry, "spp"] = spp

            rxlr = RXLR_pattern.search(seq, 0, 100)
            hdrxlr = hdRXLR_pattern.search(seq, 0, 100)
    
            if rxlr:
                end = rxlr.end()
                df.loc[entry, "type"] = "RXLR"
            
            else:
                 df.loc[entry, "type"] = "nonRXLR"
                 continue

            after_rxlr = seq[int(end):]

            eer = EER_pattern.search(after_rxlr, 0, 40)

            if eer:
                df.loc[entry, "EER"] = True
                end = eer.end()
                effector_dom = after_rxlr[int(end):]
            else:
                effector_dom = after_rxlr

            df.loc[entry, "effector_domain"] = effector_dom
            df.loc[entry, "RXLR motif"] = rxlr.group() if rxlr else None
            df.loc[entry, "EER motif"] = eer.group() if eer else None

            f.write(f">{entry}\n{effector_dom}\n")

results/Phycap_signalp_regex.fasta
results/Physoj_signalp_regex.fasta
results/Pytult_signalp_regex.fasta
results/Hyaara_signalp_regex.fasta
results/Pytaph_signalp_regex.fasta
results/Phyinf_signalp_regex.fasta
results/Phypal_signalp_regex.fasta
results/Phymeg_signalp_regex.fasta
results/Phycin_signalp_regex.fasta
results/Phycac_signalp_regex.fasta
results/Phyram_signalp_regex.fasta
results/Brelac_signalp_regex.fasta
results/Phylat_signalp_regex.fasta
results/Phypar_signalp_regex.fasta
results/Phyker_signalp_regex.fasta


In [23]:
df.reset_index(names = "entry", inplace = True)

In [24]:
df

,entry,seq,spp,type,EER,effector_domain,RXLR motif,EER motif
0,Phyca11_99641,EEDHPMAKVWRIGVTKGSALGHAETNRKPTCKPRKAAAIRAAKKGR...,Phycap,nonRXLR,NaN,NaN,NaN,NaN
1,Phyca11_100524,QPPQEEEGPVKFGPASTFPPQGDHYEPREVLRSVERPHQLFGILLA...,Phycap,nonRXLR,NaN,NaN,NaN,NaN
2,Phyca11_503479,TIYDTSSRFTGFSSDNYVRQSRADPEHVVPLVIGLLPGDFDKLERT...,Phycap,nonRXLR,NaN,NaN,NaN,NaN
3,Phyca11_525500,ASSSGSSETTTTSGTLNTLTTSCNGGCSSNSACVVVGQSSANSINC...,Phycap,nonRXLR,NaN,NaN,NaN,NaN
4,Phyca11_503502,ADTVCNPPEHNTLTSHCYGACSSSLCVNYAASTEEARNKDSSDGGF...,Phycap,nonRXLR,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
22640,Phyker_46568.5137,SNPNGVCAVDTNSINFGNTQLSSTGNAGVSAASIASDADTGKTPCP...,Phyker,nonRXLR,NaN,NaN,NaN,NaN
22641,Phyker_46568.4416,STGVEAQYRVDMRVFFRAFCPACQWFVGDPLLELVRTEEYRNIVDL...,Phyker,nonRXLR,NaN,NaN,NaN,NaN
22642,Phyker_46568.254,DRDNLPQLDELKRQRAANTSYDLADLEAPVLYGKPLQILAFAASIP...,Phyker,nonRXLR,NaN,NaN,NaN,NaN
22643,Phyker_46568.3743,EEQVTQATVDPDVGKTVVEIIGARGYEVETHKVTTTDGYILTMYRI...,Phyker,nonRXLR,NaN,NaN,NaN,NaN


In [25]:
df.to_csv("RXLR_pattern_match.tsv", sep = "\t", index = False)

In [26]:
df.groupby("type").count()

,entry,seq,spp,EER,effector_domain,RXLR motif,EER motif
type,,,,,,,
RXLR,4325,4325,4325,2717,4325,4325,2717
nonRXLR,18320,18320,18320,0,0,0,0


In [30]:
df[df["RXLR motif"].notna()].to_csv("Predicted_RXLR_effectors.tsv", sep = "\t", index = False )

In [31]:
df[df["RXLR motif"].notna()]

,entry,seq,spp,type,EER,effector_domain,RXLR motif,EER motif
38,Phyca11_503811,CISLSSAANAPTTSNGVRRLRLAVDVQEERGIADVLSGAITSMKAK...,Phycap,RXLR,True,GIADVLSGAITSMKAKVTTNSGKIVEGLKTTAQLSDDQAAKMAKIF...,RRLR,EER
41,Phyca11_5186,CNATAVSDQGKTPTVHSLDARLNDDTGDRRSLRAFKEKEEFDTEER...,Phycap,RXLR,True,DPSKADDAYKTWADKGYTLSQLSGFLTSKTKGKYDRVYNGYAIHLDY,RSLR,EER
44,Phyca11_130393,ETHITSNVQLHSEDIVRRLRFAPEQVEERGLTVPKFQDIVNNQQLA...,Phycap,RXLR,True,GLTVPKFQDIVNNQQLAWWLKRGKTTDDVFAKLKLNLAGSSIFENP...,RRLR,EER
70,Phyca11_109432,DPASVAAVHSSRVLSDEDKRFLRRHPTEHDNEERAFGQNMFAALKL...,Phycap,RXLR,True,AFGQNMFAALKLSKMKTDAEYRVKVFHRWKKHGYTADDVAKHVPAK...,RFLR,EER
98,Phyca11_129643,VNPTHDERLLRTSTSTKTSTSETEERAIDAAMIARLNDEWQAAVMR...,Phycap,RXLR,True,AIDAAMIARLNDEWQAAVMRAAYAHWFEGGKTKADVRLIMSLPPTG...,RLLR,EER
...,...,...,...,...,...,...,...,...
22581,Phyker_46568.2573,ALRSSSSSSMPEIELDYSTIRAVAANETVGYYKYQNIRFAAVPTGD...,Phyker,RXLR,NaN,FAAPEFPPQETETNTGTNLADADVDCSSTEDCLYLDVWAPANATNL...,GDLR,None
22624,Phyker_46568.713,DSPADARALNVVSNERGAKRSLRTMNEESNEVDSTDEADGTEEEDR...,Phyker,RXLR,True,ILFMGRLKNKWDLRSMRKAADKDFKKQAKELEKLDKAAKKQQKQID...,RSLR,EDR
22627,Phyker_46568.8989,GSSTATTAKPEEVYAAVFRVKDGKVTDAMLKDINQNDGSVYYLNQD...,Phyker,RXLR,NaN,IEA,RRLR,None
22635,Phyker_46568.8775,IHLTTVAWNEDKELLCHAHNKGVKVVVKHNFDDVEQLCNATARRDW...,Phyker,RXLR,NaN,RELELHPLTRNSQITFDVPWAPRGVDGRYYPWRELAGAVDFLFVMS...,QELR,None
